# Coach changes in Italian volleyball: analysis for the paper

Reproduces every number, table and figure of the manuscript from `volley_coaches_clean.zip`, the file written by
the cleaning notebook (`volley_coach_changes_clean.ipynb`).

**How to run:** Runtime, then Run all. The zip is read from Google Drive (`MyDrive/volley_coaches/`) or uploaded when asked.

**Output:** `coach_analysis_out.zip` with the tables (CSV), the figure (PDF, PNG, TIFF) and `results.json`, which holds every number quoted in the text.

## 1. Settings

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/volley_coaches/volley_coaches_clean.zip"
USE_DRIVE = True     # read the zip from Google Drive (otherwise upload it when asked)
K = 4                # matches before and after a change in the main comparison
N_CONTROLS = 5       # control teams matched to each change
WINDOW = 6           # matches shown before and after a change in Figure 1
B = 4000             # bootstrap resamples
SEED = 1
LOCAL_DATA = None    # set to a folder with the unzipped files to skip Drive and upload


## 2. Setup and data

In [ ]:
import os, json, zipfile, warnings
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import norm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
try:
    display
except NameError:
    def display(x):
        print(x.to_string() if hasattr(x, "to_string") else x)

WORK = os.environ.get("COACH_WORK", "/content/coach_analysis")
OUT = os.path.join(WORK, "out")
os.makedirs(OUT, exist_ok=True)
if LOCAL_DATA:
    DATA = LOCAL_DATA
else:
    DATA = os.path.join(WORK, "data")
    os.makedirs(DATA, exist_ok=True)
    if USE_DRIVE and not os.path.exists(ZIP_PATH):
        from google.colab import drive
        drive.mount("/content/drive")
    if not os.path.exists(ZIP_PATH):
        from google.colab import files
        print("Upload volley_coaches_clean.zip")
        up = files.upload()
        ZIP_PATH = os.path.join(WORK, list(up)[0])
        with open(ZIP_PATH, "wb") as f:
            f.write(up[list(up)[0]])
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA)

reg = pd.read_csv(os.path.join(DATA, "team_matches_regular_with_perf.csv"), low_memory=False)
reg = reg[["cid", "sex", "league", "season", "date", "team", "opp", "home", "pts_for", "pts_against",
           "sets_for", "sets_against", "coach", "coach_name", "t"]].copy()
events = pd.read_csv(os.path.join(DATA, "coach_changes.csv"))
spells = pd.read_csv(os.path.join(DATA, "coach_spells.csv"))
coverage = pd.read_csv(os.path.join(DATA, "coverage.csv"))
RES = {}   # every number quoted in the paper


def r3(x):
    return None if x is None or (isinstance(x, float) and np.isnan(x)) else round(float(x), 3)


print("Regular-season matches:", reg["cid"].nunique(), "| team-seasons:", reg.groupby(["league", "season", "team"]).ngroups,
      "| coach changes:", len(events))
RES["coverage"] = dict(matches=int(coverage["matches"].sum()), expected=int(coverage["expected"].sum()),
                       pct=r3(100 * coverage["matches"].sum() / coverage["expected"].sum()),
                       league_seasons_95=int((coverage["pct"] >= 95).sum()), league_seasons=len(coverage))


## 3. Performance, results, luck and opponent strength
For each league-season the share of rallies won by the home team is $s_m = 0.5 + h + \theta_{home} - \theta_{away} + \varepsilon_m$
(least squares, strengths summing to zero within each group of teams). The adjusted rally share of a team in a match is its share minus 0.5,
minus the home advantage at home (plus it away), plus the opponent's strength. Expected points $\hat e(s)$ are the average league points of all
matches of the same sex with a rally share in the same 0.5-point bin; luck is points minus $\hat e(s)$. The points expected from play alone are
$\hat e(0.5 + y)$, the points a team would expect with its adjusted rally share against an average opponent on neutral ground.

In [ ]:
reg["share"] = reg["pts_for"] / (reg["pts_for"] + reg["pts_against"])
reg["d"] = np.where(reg["home"] == 1, 1, -1)
H_ADV, THETA = {}, {}
for (lg, se), g in reg[reg["home"] == 1].groupby(["league", "season"]):
    teams = sorted(set(g["team"]) | set(g["opp"]))
    ix = {t: i for i, t in enumerate(teams)}
    X = np.zeros((len(g), len(teams) + 1))
    X[:, 0] = 1
    for r_, (h_, a_) in enumerate(zip(g["team"], g["opp"])):
        X[r_, ix[h_] + 1] = 1
        X[r_, ix[a_] + 1] = -1
    coef = np.linalg.lstsq(X, g["share"].values - 0.5, rcond=None)[0]
    H_ADV[(lg, se)] = coef[0]
    THETA[(lg, se)] = dict(zip(teams, coef[1:]))
reg["h"] = [H_ADV[(l, s)] for l, s in zip(reg["league"], reg["season"])]
reg["opp_str"] = [THETA[(l, s)][o] for l, s, o in zip(reg["league"], reg["season"], reg["opp"])]
reg["strength"] = [THETA[(l, s)][t] for l, s, t in zip(reg["league"], reg["season"], reg["team"])]
reg["perf"] = reg["share"] - 0.5 - reg["d"] * reg["h"] + reg["opp_str"]
reg["lpts"] = np.where((reg["sets_for"] == 3) & (reg["sets_against"] <= 1), 3,
                       np.where(reg["sets_for"] == 3, 2, np.where(reg["sets_for"] == 2, 1, 0)))
reg["share_bin"] = (reg["share"] * 200).round() / 200
EXP_LP = reg.groupby(["sex", "share_bin"])["lpts"].mean()
reg["exp_lpts"] = [EXP_LP[(a, b)] for a, b in zip(reg["sex"], reg["share_bin"])]
reg["luck"] = reg["lpts"] - reg["exp_lpts"]
EXP_ARR = {sx: EXP_LP.loc[sx].sort_index() for sx in ["M", "F"]}


def e_hat(sx, s):
    # expected points at rally share s (nearest 0.5-point bin that occurs in the data)
    e = EXP_ARR[sx]
    i = np.clip(np.searchsorted(e.index.values, s), 1, len(e) - 1)
    left, right = e.index.values[i - 1], e.index.values[i]
    j = np.where(np.abs(s - left) <= np.abs(s - right), i - 1, i)
    return e.values[j]


reg["e_play"] = np.nan
for sx in ["M", "F"]:
    m_ = reg["sex"] == sx
    reg.loc[m_, "e_play"] = e_hat(sx, (0.5 + reg.loc[m_, "perf"]).values)
reg["sched"] = reg["exp_lpts"] - reg["e_play"]   # points from the opponent and the venue

hv = np.array(list(H_ADV.values()))
RES["home_adv_pp_median"] = r3(100 * np.median(hv))
for sx in ["M", "F"]:
    s_ = reg[reg["sex"] == sx]
    fit = smf.ols("lpts ~ I(100*share)", data=s_).fit()
    RES[f"points_per_pp_{sx}"] = r3(fit.params.iloc[1])
    RES[f"sd_luck_{sx}"] = r3(s_["luck"].std())
    RES[f"sd_lpts_{sx}"] = r3(s_["lpts"].std())
    e = EXP_ARR[sx]
    RES[f"slope_near_50_{sx}"] = r3((e.loc[0.51] - e.loc[0.49]) / 2)
print({k: v for k, v in RES.items() if k != "coverage"})

# previous-season strength: strength in the same league the season before (missing for teams new to the league and
# for every team in the first season of a league in the data)
seasons = sorted(reg["season"].unique())
PREV_S = {s: seasons[i - 1] for i, s in enumerate(seasons) if i}
FIRST_SEASON = reg.groupby("league")["season"].min().to_dict()
PRESENT = set(zip(reg["league"], reg["season"], reg["team"]))
UP = {"Men A2": "Men SuperLega", "Men A3": "Men A2", "Women A2": "Women A1"}
DOWN = {"Men SuperLega": "Men A2", "Men A2": "Men A3", "Women A1": "Women A2"}


def prev_level(lg, se, team):
    p = PREV_S.get(se)
    return THETA.get((lg, p), {}).get(team, np.nan) if p else np.nan


def status(lg, se, team):
    if se == FIRST_SEASON[lg]:
        return "first season"
    p = PREV_S[se]
    if (lg, p, team) in PRESENT:
        return "same league"
    if (UP.get(lg), p, team) in PRESENT:
        return "relegated"
    if (DOWN.get(lg), p, team) in PRESENT:
        return "promoted"
    return "new"   # promoted from a league outside the data


COLS = ["perf", "lpts", "luck", "opp_str", "e_play", "sched"]
SERIES = {k: g.set_index("t")[COLS].sort_index() for k, g in reg.groupby(["league", "season", "team"])}
COACH = {k: g.set_index("t")["coach"].sort_index() for k, g in reg.groupby(["league", "season", "team"])}

# close sets (decided by two points) and tie-breaks won and lost by each team in each match
mt = pd.read_csv(os.path.join(DATA, "matches.csv"), low_memory=False)[["cid", "H", "A", "sets_str"]]
SETREC = {}
for cid, h, a, ss in zip(mt["cid"], mt["H"], mt["A"], mt["sets_str"]):
    sets = [tuple(int(v) for v in x.split("-")) for x in ss.split()]
    for team, side in [(h, 0), (a, 1)]:
        rec = [0, 0, 0, 0]
        for i, (x, y) in enumerate(sets):
            mine, theirs = (x, y) if side == 0 else (y, x)
            if abs(x - y) == 2:
                rec[0 if mine > theirs else 1] += 1
            if i == 4:
                rec[2 if mine > theirs else 3] += 1
        SETREC[(cid, team)] = rec
SETCOLS = ["close_w", "close_l", "tb_w", "tb_l"]
for i, c in enumerate(SETCOLS):
    reg[c] = [SETREC.get((cid, t), [np.nan] * 4)[i] for cid, t in zip(reg["cid"], reg["team"])]
SETS = {k: g.set_index("t")[SETCOLS].sort_index() for k, g in reg.groupby(["league", "season", "team"])}
n_sets = (reg["sets_for"] + reg["sets_against"]).sum()
RES["pct_sets_close"] = r3(100 * (reg["close_w"].sum() + reg["close_l"].sum()) / n_sets)
# share of luck variance carried by close sets: luck regressed on close sets won minus lost
reg["close_net"] = reg["close_w"] - reg["close_l"]
RES["r2_luck_close_sets"] = r3(smf.ols("luck ~ close_net", data=reg).fit().rsquared)


## 4. Data summary (Table 1)

In [ ]:
ts = reg.groupby(["league", "season", "team"]).size().rename("n").reset_index()
rows = []
for lg in ["Men SuperLega", "Men A2", "Men A3", "Women A1", "Women A2"]:
    g = reg[reg["league"] == lg]
    e = events[events["league"] == lg]
    tsl = ts[ts["league"] == lg]
    changed_ts = e.groupby(["season", "team"]).ngroups
    cv = coverage[coverage["league"] == lg]
    rows.append(dict(league=lg, seasons=f"{g['season'].min()} to {g['season'].max()}", n_seasons=g["season"].nunique(),
                     team_seasons=len(tsl), matches=g["cid"].nunique(),
                     coverage_pct=round(100 * cv["matches"].sum() / cv["expected"].sum(), 1), changes=len(e),
                     team_seasons_with_change=changed_ts,
                     pct_team_seasons_with_change=round(100 * changed_ts / len(tsl), 1),
                     median_timing=round((e["t_last"] / e["team_matches"]).median(), 2) if len(e) else np.nan,
                     interim_first=int((e["first_successor"] != e["in_name"]).sum())))
T1 = pd.DataFrame(rows)
tot = dict(league="All", seasons=f"{reg['season'].min()} to {reg['season'].max()}", n_seasons=reg["season"].nunique(),
           team_seasons=len(ts), matches=reg["cid"].nunique(),
           coverage_pct=round(100 * coverage["matches"].sum() / coverage["expected"].sum(), 1), changes=len(events),
           team_seasons_with_change=events.groupby(["league", "season", "team"]).ngroups,
           pct_team_seasons_with_change=round(100 * events.groupby(["league", "season", "team"]).ngroups / len(ts), 1),
           median_timing=round((events["t_last"] / events["team_matches"]).median(), 2),
           interim_first=int((events["first_successor"] != events["in_name"]).sum()))
T1 = pd.concat([T1, pd.DataFrame([tot])], ignore_index=True)
display(T1)
T1.to_csv(os.path.join(OUT, "table1_data.csv"), index=False)
RES["table1"] = T1.to_dict(orient="records")
RES["n_changes"] = len(events)
RES["n_first_changes"] = int(events["season_opener"].sum())
RES["n_overlap_changes"] = int(events["overlap"].sum())
RES["n_teams_two_or_more_changes"] = int((events.groupby(["league", "season", "team"]).size() > 1).sum())
RES["n_first_changes_by_match3"] = int((events["season_opener"] & (events["t_last"] < K)).sum())
RES["n_team_seasons_no_opener"] = int((~events.groupby(["league", "season", "team"])["season_opener"].any()).sum())
RES["pct_covid_seasons"] = coverage[coverage["season"] == "2019/20"][["league", "pct"]].to_dict(orient="records")


## 5. Is luck persistent? (Table 3)

In [ ]:
ev_keys = set(zip(events["league"], events["season"], events["team"]))
spell_n = spells[spells["matches"] >= 3].groupby(["league", "season", "team"]).size()


def split_halves(split):
    rows = []
    for key, g in reg.groupby(["league", "season", "team"]):
        g = g.sort_values("t")
        n = len(g)
        if n < 10:
            continue
        second = (np.arange(n) >= n // 2) if split == "half" else (np.arange(n) % 2 == 1)
        r = dict(league=key[0], season=key[1], team=key[2], kept=key not in ev_keys and spell_n.get(key, 1) <= 1)
        for p, m in [(1, ~second), (2, second)]:
            d = g[m]
            r[f"perf{p}"] = 100 * d["perf"].mean()
            r[f"lpts{p}"] = d["lpts"].mean()
            r[f"luck{p}"] = d["luck"].mean()
            for nm, w, l in [("close", "close_w", "close_l"), ("tb", "tb_w", "tb_l")]:
                nn = d[w].sum() + d[l].sum()
                r[f"{nm}{p}"] = 100 * d[w].sum() / nn if nn else np.nan
                r[f"n_{nm}{p}"] = nn
        rows.append(r)
    return pd.DataFrame(rows)


def corr_boot(D, v, given=False, seed=SEED):
    d = D.dropna(subset=[v + "1", v + "2"]).reset_index(drop=True)
    lg = pd.get_dummies(d["league"], drop_first=True).to_numpy(float)
    X = [np.column_stack([np.ones(len(d)), d[f"perf{p}"].to_numpy(float), lg]) for p in (1, 2)]
    Y = [d[f"{v}{p}"].to_numpy(float) for p in (1, 2)]

    def r(ix):
        a, b = Y[0][ix], Y[1][ix]
        if given:
            a = a - X[0][ix] @ np.linalg.lstsq(X[0][ix], a, rcond=None)[0]
            b = b - X[1][ix] @ np.linalg.lstsq(X[1][ix], b, rcond=None)[0]
        return np.corrcoef(a, b)[0, 1]
    rng = np.random.default_rng(seed)
    bs = [r(rng.integers(0, len(d), len(d))) for _ in range(B)]
    return r(np.arange(len(d))), np.percentile(bs, 2.5), np.percentile(bs, 97.5), len(d)


HALF, ODD = split_halves("half"), split_halves("oddeven")
P_ROWS = [("Adjusted rally share", "perf", False), ("League points", "lpts", False), ("Luck", "luck", False),
          ("Close sets won", "close", False), ("Close sets won, given rally share", "close", True),
          ("Tie-breaks won", "tb", False), ("Tie-breaks won, given rally share", "tb", True)]
rows = []
for name, v, given in P_ROWS:
    a, b = corr_boot(HALF, v, given), corr_boot(ODD, v, given)
    rows.append(dict(measure=name, half_r=a[0], half_lo=a[1], half_hi=a[2], half_n=a[3],
                     oddeven_r=b[0], oddeven_lo=b[1], oddeven_hi=b[2], oddeven_n=b[3]))
T3 = pd.DataFrame(rows)
display(T3.round(3))
T3.to_csv(os.path.join(OUT, "table3_persistence.csv"), index=False)
m_l = smf.ols("lpts2 ~ perf1 + luck1 + C(league)", data=HALF).fit(cov_type="HC1")
ci_l = m_l.conf_int().loc["luck1"]
r2_perf = smf.ols("lpts2 ~ perf1 + C(league)", data=HALF).fit().rsquared
r2_lpts = smf.ols("lpts2 ~ lpts1 + C(league)", data=HALF).fit().rsquared
kept_r = corr_boot(HALF[HALF["kept"]], "luck")
RES["persistence"] = dict(
    n_team_seasons=len(HALF), table3=T3.to_dict(orient="records"),
    luck1_on_lpts2_per_0_1=[r3(0.1 * m_l.params["luck1"]), r3(0.1 * ci_l[0]), r3(0.1 * ci_l[1])],
    r2_lpts2_from_perf1=r3(r2_perf), r2_lpts2_from_lpts1=r3(r2_lpts),
    luck_kept_coach=dict(r=r3(kept_r[0]), lo=r3(kept_r[1]), hi=r3(kept_r[2]), n=kept_r[3]))
print(RES["persistence"]["luck1_on_lpts2_per_0_1"], r2_perf, r2_lpts, RES["persistence"]["luck_kept_coach"])


## 6. What triggers a change (Table 4)

In [ ]:
first_t = events[events["season_opener"]].set_index(["league", "season", "team"])["t_last"]
n_spells = spells[spells["matches"] >= 3].groupby(["league", "season", "team"]).size()
hz = []
for key, s in SERIES.items():
    changed = key in first_t.index
    if not changed and n_spells.get(key, 1) > 1:
        continue        # several coaches but no clean change (interim spells): left out
    n = s.index.max()
    end = first_t[key] if changed else n - 3
    pl = prev_level(*key)
    st = status(*key)
    for t in range(K, end + 1):
        w = s.reindex(range(t - K + 1, t + 1))
        if t not in s.index or w["perf"].notna().sum() < 3:
            continue
        hz.append(dict(league=key[0], season=key[1], team=key[2], unit="|".join(key), t=t,
                       y=int(changed and t == end),
                       perf4=100 * w["perf"].mean(), luck4=10 * w["luck"].mean(), opp4=100 * w["opp_str"].mean(),
                       lpts4=w["lpts"].mean(), perf_last=100 * s.at[t, "perf"], luck_last=10 * s.at[t, "luck"],
                       prior=100 * s["perf"].reindex(range(1, t - K + 1)).mean(), prev=100 * pl,
                       first_season=int(st == "first season")))
HZ = pd.DataFrame(hz)
HZ["new_league"] = (HZ["prev"].isna() & (HZ["first_season"] == 0)).astype(int)
HZ["prev0"] = HZ["prev"].fillna(0)
HZ["prior_missing"] = HZ["prior"].isna().astype(int)
HZ["prior0"] = HZ["prior"].fillna(0)
HZ["t_c"] = HZ["t"] - HZ["t"].mean()
groups = HZ["unit"].astype("category").cat.codes
f_main = ("y ~ perf4 + luck4 + perf_last + luck_last + opp4 + prior0 + prior_missing + prev0 + new_league + first_season"
          " + t_c + I(t_c**2) + C(league)")
m_main = smf.logit(f_main, data=HZ).fit(disp=0, cov_type="cluster", cov_kwds={"groups": groups})
f_pts = "y ~ perf4 + lpts4 + prior0 + prior_missing + prev0 + new_league + first_season + t_c + I(t_c**2) + C(league)"
m_pts = smf.logit(f_pts, data=HZ).fit(disp=0, cov_type="cluster", cov_kwds={"groups": groups})
f_pts_full = ("y ~ perf4 + lpts4 + perf_last + luck_last + opp4 + prior0 + prior_missing + prev0 + new_league + first_season"
              " + t_c + I(t_c**2) + C(league)")
m_pts_full = smf.logit(f_pts_full, data=HZ).fit(disp=0, cov_type="cluster", cov_kwds={"groups": groups})
LABELS = {"perf4": "Adjusted rally share, last 4 matches (per 1 pp)",
          "luck4": "Luck, last 4 matches (per 0.1 points per match)",
          "perf_last": "Adjusted rally share, last match (per 1 pp)",
          "luck_last": "Luck, last match (per 0.1 points)",
          "opp4": "Opponent strength, last 4 matches (per 1 pp)",
          "prior0": "Form earlier in the season (per 1 pp)",
          "prev0": "Previous-season strength (per 1 pp)",
          "new_league": "New to the league",
          "first_season": "First season of the league in the data"}
ci = m_main.conf_int()
T4 = pd.DataFrame([dict(predictor=LABELS[v], odds_ratio=np.exp(m_main.params[v]),
                        ci_low=np.exp(ci.loc[v, 0]), ci_high=np.exp(ci.loc[v, 1]), p_value=m_main.pvalues[v])
                   for v in LABELS])
print(f"Team-match observations at risk: {len(HZ)}, first changes: {int(HZ['y'].sum())}, baseline rate: {100 * HZ['y'].mean():.2f}%")
display(T4.round(3))
T4.to_csv(os.path.join(OUT, "table4_triggers.csv"), index=False)
RES["hazard_n_obs"] = len(HZ)
RES["hazard_n_events"] = int(HZ["y"].sum())
RES["hazard_n_units"] = int(HZ["unit"].nunique())
RES["hazard_rate_pct"] = r3(100 * HZ["y"].mean())
RES["hazard_main"] = {v: dict(OR=r3(np.exp(m_main.params[v])), lo=r3(np.exp(ci.loc[v, 0])), hi=r3(np.exp(ci.loc[v, 1])),
                              p=r3(m_main.pvalues[v])) for v in LABELS}
for nm, mm in [("hazard_points_vs_play", m_pts), ("hazard_points_vs_play_full", m_pts_full)]:
    ci2 = mm.conf_int()
    RES[nm] = {v: dict(OR=r3(np.exp(mm.params[v])), lo=r3(np.exp(ci2.loc[v, 0])), hi=r3(np.exp(ci2.loc[v, 1])),
                       p=r3(mm.pvalues[v])) for v in ["perf4", "lpts4"]}
print(RES["hazard_points_vs_play"], RES["hazard_points_vs_play_full"])
base = HZ[HZ["y"] == 1][["perf4", "perf_last", "luck_last", "opp4", "prior0", "prev0"]].median()
pred_rows = []
for q in [0.1, 0.9]:
    row = dict(base)
    row.update(luck4=HZ["luck4"].quantile(q), prior_missing=0, new_league=0, first_season=0, t_c=0, league="Men SuperLega")
    pred_rows.append(row)
pp_ = m_main.predict(pd.DataFrame(pred_rows))
RES["hazard_pred_luck_p10_p90_pct"] = [r3(100 * x) for x in pp_]
RES["luck4_p10_p90"] = [r3(HZ["luck4"].quantile(0.1) / 10), r3(HZ["luck4"].quantile(0.9) / 10)]
RES["sd_prev_strength_pp"] = r3(HZ.drop_duplicates("unit")["prev"].std())
print("Predicted probability (luck at 10th vs 90th percentile):", RES["hazard_pred_luck_p10_p90_pct"], RES["luck4_p10_p90"])


## 7. Before and after a change: matched comparison (Table 5, Figure 1)
A candidate control is a team at a match t such that the same head coach led it in all matches from t-3 to t+4, all before the
team's first coach change of the season (teams that change coach later can serve as controls before their change; a team is never
its own control). For each change the five closest candidates of the same sex, within three matches of the change and with the same
previous-season status, are taken on the adjusted rally share before the change, the form earlier in the season and the
previous-season strength.

Intervals resample the changes and, independently, the control team-seasons (a pigeonhole bootstrap), because a control team-season
can serve several changes.

In [ ]:
first_change_t = events.groupby(["league", "season", "team"])["t_last"].min().to_dict()


def pre_post(s, t, col, k=K, rest=False):
    a = s[col].reindex(range(t - k + 1, t + 1))
    b = s[col].loc[s.index > t] if rest else s[col].reindex(range(t + 1, t + k + 1))
    return a.mean(), b.mean(), a.notna().sum(), b.notna().sum()


def at_k(s, t, col, k):
    return s[col].get(t + k + 1 if k < 0 else t + k, np.nan)


def candidates(k=K, var="perf"):
    out = []
    for key, s in SERIES.items():
        co = COACH[key]
        tmax = s.index.max() - k
        if key in first_change_t:
            tmax = min(tmax, first_change_t[key] - k)
        pl = prev_level(*key)
        for t in range(k, tmax + 1):
            a, b, na, nb = pre_post(s, t, var, k)
            if na < 3 or nb < 3:
                continue
            cw = co.reindex(range(t - k + 1, t + k + 1)).dropna()
            if cw.nunique() > 1:
                continue
            out.append(dict(key=key, league=key[0], sex="F" if key[0].startswith("Women") else "M", t=t, pre=a,
                            prior=s[var].reindex(range(1, t - k + 1)).mean(), prev=pl, has_prev=pd.notna(pl)))
    return pd.DataFrame(out)


KS = [k for k in range(-WINDOW, WINDOW + 1) if k != 0]


def matched(ev, C, weights=None, mode="same_status", k=K, nc=N_CONTROLS, curves=False, var="perf", rest=False):
    weights = weights or {"pre": 1, "prior": 1, "prev": 1}
    sd = {c: C[c].std() for c in ["pre", "prior", "prev"]}
    rows = []
    for e in ev.itertuples():
        key = (e.league, e.season, e.team)
        s = SERIES.get(key)
        if s is None:
            continue
        a, b, na, nb = pre_post(s, e.t_last, var, k)
        if na < 3 or nb < 3:
            continue
        x = dict(pre=a, prior=s[var].reindex(range(1, e.t_last - k + 1)).mean(), prev=prev_level(*key))
        sx = "F" if e.league.startswith("Women") else "M"
        pool = C[(C["sex"] == sx) & ((C["t"] - e.t_last).abs() <= 3)]
        pool = pool[[kk != key for kk in pool["key"]]]
        if mode == "require_prev":
            if pd.isna(x["prev"]):
                continue
            pool = pool[pool["has_prev"]]
        elif mode == "same_status":
            pool = pool[pool["has_prev"] == pd.notna(x["prev"])]
        dist = pd.Series(0.0, index=pool.index)
        for c, w in weights.items():
            if w and pd.notna(x[c]):
                dist += w * ((pool[c] - x[c]).abs() / sd[c]).fillna(1.0)
        ctrl = pool.loc[dist.nsmallest(nc).index]
        r = dict(league=e.league, sex=sx, season=e.season, team=e.team, t_last=e.t_last,
                 opener=getattr(e, "season_opener", True), overlap=getattr(e, "overlap", False),
                 status=status(*key), prev=x["prev"], prior=x["prior"], pre_match=x["pre"],
                 c_prev=ctrl["prev"].mean(), c_prior=ctrl["prior"].mean(), c_pre_match=ctrl["pre"].mean(),
                 ctrl=list(zip(ctrl["key"], ctrl["t"])), n_after=int(s.index.max() - e.t_last) if rest else k)
        for col in COLS:
            a, b, _, _ = pre_post(s, e.t_last, col, k, rest)
            r[col + "_pre"], r[col + "_post"] = a, b
            r[col + "_cpre_list"] = [pre_post(SERIES[c.key], c.t, col, k, rest)[0] for c in ctrl.itertuples()]
            r[col + "_cpost_list"] = [pre_post(SERIES[c.key], c.t, col, k, rest)[1] for c in ctrl.itertuples()]
            r[col + "_cpre"], r[col + "_cpost"] = np.nanmean(r[col + "_cpre_list"]), np.nanmean(r[col + "_cpost_list"])
        if curves:
            for col in ["perf", "lpts"]:
                for kk in KS:
                    r[f"{col}_k{kk}"] = at_k(s, e.t_last, col, kk)
                    r[f"{col}_ck_list{kk}"] = [at_k(SERIES[c.key], c.t, col, kk) for c in ctrl.itertuples()]
                    r[f"{col}_ck{kk}"] = np.nanmean(r[f"{col}_ck_list{kk}"])
        rows.append(r)
    return pd.DataFrame(rows)


SC = {"perf": 100, "lpts": 1, "luck": 1, "opp_str": 100, "e_play": 1, "sched": 1}
NAMES = {"perf": "Adjusted rally share (pp)", "lpts": "League points per match", "luck": "Luck (points per match)",
         "opp_str": "Opponent strength (pp)", "e_play": "Points expected from play", "sched": "Points from opponents and venue"}


def pigeonhole(R, treated, control_lists, stat="mean", reps=B, seed=SEED):
    # estimate and 95% interval of mean_e [treated_e - mean_c control_ec], resampling changes and control team-seasons
    t_ = np.asarray(treated, float)
    keys = sorted({kk for cl in R["ctrl"] for kk, tt in cl})
    kid = {k: i for i, k in enumerate(keys)}
    nc = max(len(cl) for cl in R["ctrl"])
    idx = np.zeros((len(R), nc), int)
    val = np.full((len(R), nc), np.nan)
    for i, (cl, vals) in enumerate(zip(R["ctrl"], control_lists)):
        for j, ((kk, tt), v) in enumerate(zip(cl, vals)):
            idx[i, j] = kid[kk]
            val[i, j] = v
    ok = ~np.isnan(val)
    v0 = np.where(ok, val, 0.0)

    def est(wc, rows):
        W = wc[idx] * ok
        den = W.sum(1)
        cm = np.where(den > 0, (W * v0).sum(1) / np.where(den > 0, den, 1), np.nan)
        d = (t_ - cm)[rows]
        return np.nanmean(d)
    point = est(np.ones(len(keys)), np.arange(len(R)))
    rng = np.random.default_rng(seed)
    bs = np.empty(reps)
    for b in range(reps):
        wc = np.bincount(rng.integers(0, len(keys), len(keys)), minlength=len(keys)).astype(float)
        rows = rng.integers(0, len(R), len(R))
        bs[b] = est(wc, rows)
    return point, np.nanpercentile(bs, 2.5), np.nanpercentile(bs, 97.5), np.nanstd(bs, ddof=1)


def did(R, col, reps=B):
    sc = SC[col]
    tr = sc * (R[col + "_post"] - R[col + "_pre"])
    cl = [[sc * (b - a) for a, b in zip(pa, pb)] for pa, pb in zip(R[col + "_cpre_list"], R[col + "_cpost_list"])]
    return pigeonhole(R, tr, cl, reps=reps)


def gap(R, col, when, reps=B):
    sc = SC[col]
    tr = sc * R[f"{col}_{when}"]
    cl = [[sc * v for v in L] for L in R[f"{col}_c{when}_list"]]
    return pigeonhole(R, tr, cl, reps=reps)


def table5(R):
    rows = []
    for col in ["perf", "lpts", "luck", "opp_str"]:
        sc = SC[col]
        gb, ga, dd = gap(R, col, "pre"), gap(R, col, "post"), did(R, col)
        rows.append(dict(measure=NAMES[col], changed_before=sc * R[col + "_pre"].mean(), changed_after=sc * R[col + "_post"].mean(),
                         control_before=sc * R[col + "_cpre"].mean(), control_after=sc * R[col + "_cpost"].mean(),
                         gap_before=gb[0], gap_before_lo=gb[1], gap_before_hi=gb[2],
                         gap_after=ga[0], gap_after_lo=ga[1], gap_after_hi=ga[2],
                         did=dd[0], did_lo=dd[1], did_hi=dd[2]))
    return pd.DataFrame(rows)


def set_counts(R, w, l, k=K):
    rows = []
    for r in R.itertuples():
        s = SETS[(r.league, r.season, r.team)]
        out = {"cw": [], "cl": []}
        for post in (0, 1):
            rng_ = range(r.t_last + 1, r.t_last + k + 1) if post else range(r.t_last - k + 1, r.t_last + 1)
            d = s.reindex(rng_)
            out[f"w{post}"], out[f"l{post}"] = d[w].sum(), d[l].sum()
        for key, t in r.ctrl:
            cwl = []
            for post in (0, 1):
                rng_c = range(t + 1, t + k + 1) if post else range(t - k + 1, t + 1)
                dc = SETS[key].reindex(rng_c)
                cwl += [dc[w].sum(), dc[l].sum()]
            out["cw"].append(cwl)
        rows.append(out)
    return rows


def rate_row(R, S, name, reps=B, seed=SEED):
    keys = sorted({kk for cl in R["ctrl"] for kk, tt in cl})
    kid = {k: i for i, k in enumerate(keys)}
    T = np.array([[s["w0"], s["l0"], s["w1"], s["l1"]] for s in S], float)
    nc = max(len(s["cw"]) for s in S)
    idx = np.zeros((len(S), nc), int)
    Cc = np.zeros((len(S), nc, 4))
    msk = np.zeros((len(S), nc))
    for i, (s, cl) in enumerate(zip(S, R["ctrl"])):
        for j, ((kk, tt), c4) in enumerate(zip(cl, s["cw"])):
            idx[i, j] = kid[kk]
            Cc[i, j] = c4
            msk[i, j] = 1

    def stats(wc, rows):
        W = wc[idx] * msk
        den = W.sum(1, keepdims=True)
        cavg = (W[:, :, None] * Cc).sum(1) / np.where(den > 0, den, 1)
        Tr, Cr = T[rows].sum(0), cavg[rows].sum(0)
        rt = [Tr[0] / (Tr[0] + Tr[1]), Tr[2] / (Tr[2] + Tr[3])]
        rc = [Cr[0] / (Cr[0] + Cr[1]), Cr[2] / (Cr[2] + Cr[3])]
        return 100 * np.array(rt + rc + [rt[0] - rc[0], rt[1] - rc[1], (rt[1] - rc[1]) - (rt[0] - rc[0])])
    est = stats(np.ones(len(keys)), np.arange(len(S)))
    rng = np.random.default_rng(seed)
    bs = np.array([stats(np.bincount(rng.integers(0, len(keys), len(keys)), minlength=len(keys)).astype(float),
                         rng.integers(0, len(S), len(S))) for _ in range(reps)])
    lo, hi = np.percentile(bs, 2.5, axis=0), np.percentile(bs, 97.5, axis=0)
    return dict(measure=name, changed_before=est[0], changed_after=est[1], control_before=est[2], control_after=est[3],
                gap_before=est[4], gap_before_lo=lo[4], gap_before_hi=hi[4], gap_after=est[5], gap_after_lo=lo[5],
                gap_after_hi=hi[5], did=est[6], did_lo=lo[6], did_hi=hi[6]), int(T[:, :2].sum()), int(T[:, 2:].sum())


CAND = candidates()
R_main = matched(events, CAND, curves=True)
T5 = table5(R_main)
rc, n_close_pre, n_close_post = rate_row(R_main, set_counts(R_main, "close_w", "close_l"), "Close sets won (%)")
rt_, n_tb_pre, n_tb_post = rate_row(R_main, set_counts(R_main, "tb_w", "tb_l"), "Tie-breaks won (%)")
T5 = pd.concat([T5, pd.DataFrame([rc, rt_])], ignore_index=True)
display(T5.round(3))
T5.to_csv(os.path.join(OUT, "table5_before_after.csv"), index=False)
R_main.drop(columns=[c for c in R_main.columns if c.endswith("_list") or "_ck_list" in c or c == "ctrl"]).to_csv(
    os.path.join(OUT, "events_matched.csv"), index=False)
RES["n_candidates"] = len(CAND)
RES["n_candidate_team_seasons_changing_later"] = int(len({k for k in CAND["key"] if k in first_change_t}))
RES["n_matched"] = len(R_main)
RES["close_sets_n"] = dict(changed_before=n_close_pre, changed_after=n_close_post)
RES["tiebreaks_n"] = dict(changed_before=n_tb_pre, changed_after=n_tb_post)
RES["balance"] = dict(pre=r3(100 * R_main["pre_match"].mean()), pre_ctrl=r3(100 * R_main["c_pre_match"].mean()),
                      prev=r3(100 * R_main["prev"].mean()), prev_ctrl=r3(100 * R_main["c_prev"].mean()),
                      prior=r3(100 * R_main["prior"].mean()), prior_ctrl=r3(100 * R_main["c_prior"].mean()),
                      n_prev=int(R_main["prev"].notna().sum()))
cu = Counter(kk for cl in R_main["ctrl"] for kk, tt in cl)
RES["control_reuse"] = dict(slots=int(sum(cu.values())), team_seasons=len(cu), max_use=int(max(cu.values())))
RES["table5_before_after"] = T5.to_dict(orient="records")
print("Balance:", RES["balance"], "| control reuse:", RES["control_reuse"])

# decomposition of the gain in league points: play, opponents and venue, luck
DEC = []
for col in ["lpts", "e_play", "sched", "luck"]:
    a = did(R_main, col)
    DEC.append(dict(part=NAMES.get(col, col), did=a[0], lo=a[1], hi=a[2]))
DEC = pd.DataFrame(DEC)
display(DEC.round(3))
DEC.to_csv(os.path.join(OUT, "decomposition.csv"), index=False)
RES["decomposition"] = DEC.to_dict(orient="records")

# match by match (Figure 1)
per_k = []
for kk in KS:
    a = pigeonhole(R_main, 100 * R_main[f"perf_k{kk}"], [[100 * v for v in L] for L in R_main[f"perf_ck_list{kk}"]])
    b = pigeonhole(R_main, R_main[f"lpts_k{kk}"], R_main[f"lpts_ck_list{kk}"])
    per_k.append(dict(k=kk, perf=a[0], perf_lo=a[1], perf_hi=a[2], lpts=b[0], lpts_lo=b[1], lpts_hi=b[2]))
PK = pd.DataFrame(per_k)
display(PK.round(3))
PK.to_csv(os.path.join(OUT, "per_match_differences.csv"), index=False)
RES["per_k"] = PK.to_dict(orient="records")

plt.rcParams.update({"font.size": 10, "font.family": "DejaVu Sans"})
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (col, sc, ylab, ttl) in zip(axes, [("perf", 100, "Adjusted rally share (pp)", "(a) Performance"),
                                           ("lpts", 1, "League points per match", "(b) Results")]):
    for pref, colr, mk, lab in [(f"{col}_k", "#2a78d6", "o", "Coach changed"),
                                (f"{col}_ck", "#eb6834", "s", "Matched control teams")]:
        Dm = pd.DataFrame({kk: sc * R_main[f"{pref}{kk}"] for kk in KS})
        mu, se = Dm.mean(), Dm.std() / np.sqrt(Dm.notna().sum())
        ax.fill_between(KS, mu - 1.96 * se, mu + 1.96 * se, color=colr, alpha=0.15, linewidth=0)
        ax.plot(KS, mu, marker=mk, color=colr, linewidth=1.8, markersize=5, label=lab)
    ax.axvline(0, color="#9a9a96", linestyle="--", linewidth=1)
    ax.set_xticks(KS)
    ax.set_xlabel("Matches before (−) and after (+) the change")
    ax.set_ylabel(ylab)
    ax.set_title(ttl, loc="left", fontsize=10)
    for s_ in ["top", "right"]:
        ax.spines[s_].set_visible(False)
    ax.grid(axis="y", color="#e6e5e0", linewidth=0.8)
axes[0].legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "figure1_event_study.pdf"))
plt.savefig(os.path.join(OUT, "figure1_event_study.png"), dpi=200)
plt.savefig(os.path.join(OUT, "figure1_event_study.tif"), dpi=600, pil_kwargs={"compression": "tiff_lzw"})
plt.close("all")


## 8. Robustness, subgroups and a placebo that copies how clubs choose the moment (Table 6)

In [ ]:
rob = []


def add(label, R, reps=B):
    a, b = did(R, "perf", reps), did(R, "lpts", reps)
    rob.append(dict(analysis=label, n=len(R), perf_did=a[0], perf_lo=a[1], perf_hi=a[2],
                    lpts_did=b[0], lpts_lo=b[1], lpts_hi=b[2]))


add("Main comparison", R_main)
add("First change of the season", R_main[R_main["opener"]])
add("Without changes with overlapping names", R_main[~R_main["overlap"]])
add("Same league in the previous season", R_main[R_main["status"] == "same league"])
add("New to the league", R_main[R_main["status"].isin(["promoted", "relegated", "new"])])
add("First season of the league in the data", R_main[R_main["status"] == "first season"])
add("Without previous-season strength", matched(events, CAND, {"pre": 1, "prior": 0.5}, mode="all"))
add("Men", R_main[R_main["sex"] == "M"])
add("Women", R_main[R_main["sex"] == "F"])
add("Top division (SuperLega, women's A1)", R_main[R_main["league"].isin(["Men SuperLega", "Women A1"])])
add("Lower divisions", R_main[~R_main["league"].isin(["Men SuperLega", "Women A1"])])
add("Change by match 10", R_main[R_main["t_last"] <= 10])
add("Change after match 10", R_main[R_main["t_last"] > 10])
add("Three control teams per change", matched(events, CAND, nc=3))
add("Ten control teams per change", matched(events, CAND, nc=10))
add("Windows of three matches", matched(events, candidates(3), k=3))
add("Windows of six matches", matched(events, candidates(6), k=6))
add("Controls only from teams that kept their coach all season",
    matched(events, CAND[[k not in first_change_t and spell_n.get(k, 1) <= 1 for k in CAND["key"]]]))
R_pts = matched(events, candidates(var="lpts"), var="lpts")
add("Matched on league points", R_pts)
R_rest = matched(events, CAND, rest=True)
add("Rest of the season after the change", R_rest)
RES["rest_median_matches"] = r3(R_rest["n_after"].median())

# placebo: for each real first change, a team at a valid control point with a similar predicted hazard
# (one of the ten closest linear predictors, same sex, within three matches) is treated as if its coach had changed
p_ = m_main.predict(HZ)
HZ["xb"] = np.log(p_ / (1 - p_))
HZ["sex"] = np.where(HZ["league"].str.startswith("Women"), "F", "M")
HZ["key"] = list(zip(HZ["league"], HZ["season"], HZ["team"]))
valid = set(zip(CAND["key"], CAND["t"]))
pool_all = HZ[(HZ["y"] == 0).to_numpy() & np.array([(k, t) in valid for k, t in zip(HZ["key"], HZ["t"])])].copy()
real = HZ[HZ["y"] == 1].set_index("key")
N_PLACEBO = 50
rng = np.random.default_rng(SEED)
pl = []
for rep in range(N_PLACEBO):
    pseudo = []
    for e in R_main[R_main["opener"]].itertuples():
        k = (e.league, e.season, e.team)
        if k not in real.index:
            continue
        xb_e = real.loc[[k]]["xb"].iloc[0]
        pool = pool_all[(pool_all["sex"] == e.sex) & ((pool_all["t"] - e.t_last).abs() <= 3)]
        j = (pool["xb"] - xb_e).abs().nsmallest(10).index
        pick = pool.loc[rng.choice(j)]
        pseudo.append(dict(league=pick["league"], season=pick["season"], team=pick["team"], t_last=int(pick["t"]),
                           season_opener=True, overlap=False))
    RP = matched(pd.DataFrame(pseudo), CAND)
    pl.append(dict(perf=did(RP, "perf", reps=1)[0], lpts=did(RP, "lpts", reps=1)[0], luck=did(RP, "luck", reps=1)[0],
                   e_play=did(RP, "e_play", reps=1)[0], sched=did(RP, "sched", reps=1)[0], n=len(RP)))
PL = pd.DataFrame(pl)
rob.append(dict(analysis=f"Placebo: similar predicted risk, coach kept (mean of {N_PLACEBO} draws)", n=int(PL["n"].iloc[0]),
                perf_did=PL["perf"].mean(), perf_lo=PL["perf"].quantile(0.025), perf_hi=PL["perf"].quantile(0.975),
                lpts_did=PL["lpts"].mean(), lpts_lo=PL["lpts"].quantile(0.025), lpts_hi=PL["lpts"].quantile(0.975)))
RES["placebo"] = {c: dict(mean=r3(PL[c].mean()), lo=r3(PL[c].quantile(0.025)), hi=r3(PL[c].quantile(0.975)))
                  for c in ["perf", "lpts", "luck", "e_play", "sched"]}
RES["placebo"]["n"] = int(PL["n"].iloc[0])
# the first changes the placebo copies (those in the hazard model), for a like-for-like comparison
R_src = R_main[R_main["opener"] & np.array([(l, se, t) in real.index for l, se, t in zip(R_main["league"], R_main["season"], R_main["team"])])]
RES["placebo_source"] = {c: [r3(x) for x in did(R_src, c)[:3]] for c in ["perf", "lpts", "e_play", "sched", "luck"]}
RES["placebo_source"]["n"] = len(R_src)
T6 = pd.DataFrame(rob)
display(T6.round(3))
T6.to_csv(os.path.join(OUT, "table6_robustness.csv"), index=False)
RES["table6_robustness"] = T6.to_dict(orient="records")
RES["first_changes_decomposition"] = {c: r3(did(R_main[R_main["opener"]], c)[0]) for c in ["lpts", "e_play", "sched", "luck"]}
RES["pts_matched_decomposition"] = {c: [r3(x) for x in did(R_pts, c)[:3]] for c in ["perf", "lpts", "e_play", "sched", "luck", "opp_str"]}
GAPS = {w: {c: gap(R_main, c, w)[:3] for c in ["lpts", "e_play", "sched", "luck"]} for w in ["pre", "post"]}
RES["gap_decomposition"] = {w: {c: [r3(x) for x in GAPS[w][c]] for c in GAPS[w]} for w in GAPS}
pd.DataFrame([dict(window=w, part=NAMES[c], gap=v[0], lo=v[1], hi=v[2]) for w in GAPS for c, v in GAPS[w].items()]).to_csv(
    os.path.join(OUT, "gap_decomposition.csv"), index=False)
print(RES["placebo"], RES["first_changes_decomposition"])


## 9. Coaching careers, type of new coach, and the following season

In [ ]:
LEVEL = {"Men SuperLega": 1, "Men A2": 2, "Men A3": 3, "Women A1": 1, "Women A2": 2}
sp = spells[spells["matches"] >= 3].copy()
sp["ys"] = sp["season"].str[:4].astype(int)
sp["sx"] = np.where(sp["league"].str.startswith("Women"), "F", "M")
sp["level"] = sp["league"].map(LEVEL)
ev = events.copy()
ev["ys"] = ev["season"].str[:4].astype(int)
ev["sx"] = np.where(ev["league"].str.startswith("Women"), "F", "M")
ev["level"] = ev["league"].map(LEVEL)
last_season = sp["ys"].max()


def coach_info(e):
    before = sp[(sp["coach"] == e.in_coach) & (sp["sx"] == e.sx) & (sp["ys"] < e.ys)]
    fired_earlier = ev[(ev["out_coach"] == e.in_coach) & (ev["sx"] == e.sx) & (ev["season"] == e.season)
                       & (ev["team"] != e.team) & (ev["date_last"] < e.date_next)]
    later_same = sp[(sp["coach"] == e.out_coach) & (sp["sx"] == e.sx) & (sp["ys"] == e.ys) & (sp["team"] != e.team)
                    & (sp["date_first"] > e.date_last)]
    nxt = sp[(sp["coach"] == e.out_coach) & (sp["sx"] == e.sx) & (sp["ys"] == e.ys + 1)]
    kept = sp[(sp["coach"] == e.in_coach) & (sp["sx"] == e.sx) & (sp["ys"] == e.ys + 1) & (sp["team"] == e.team)]
    return pd.Series(dict(in_experienced=len(before) > 0, in_recycled=len(fired_earlier) > 0,
                          interim=e.first_successor != e.in_name,
                          out_rehired_same=len(later_same) > 0, out_next=len(nxt) > 0,
                          out_next_same_level=bool(len(nxt) and nxt["level"].min() <= e.level),
                          in_kept_next=len(kept) > 0))


ev = pd.concat([ev, ev.apply(coach_info, axis=1)], axis=1)
done = ev[ev["ys"] < last_season]
RES["careers"] = dict(
    n=len(ev), new_experienced=int(ev["in_experienced"].sum()), new_recycled=int(ev["in_recycled"].sum()),
    interim=int(ev["interim"].sum()), n_with_next=len(done),
    pct_out_rehired_same=r3(100 * ev["out_rehired_same"].mean()),
    pct_out_next=r3(100 * done["out_next"].mean()), pct_out_next_same_level=r3(100 * done["out_next_same_level"].mean()),
    pct_in_kept_next=r3(100 * done["in_kept_next"].mean()))
print(RES["careers"])
Rm = R_main.merge(ev[["league", "season", "team", "t_last", "in_experienced", "in_recycled", "interim"]],
                  on=["league", "season", "team", "t_last"], how="left")
types_ = []
for label, m in [("Experienced new coach", Rm["in_experienced"]), ("New coach without earlier spells", ~Rm["in_experienced"]),
                 ("Caretaker first", Rm["interim"]), ("Direct appointment", ~Rm["interim"])]:
    a, b = did(Rm[m.values], "perf"), did(Rm[m.values], "lpts")
    types_.append(dict(group=label, n=int(m.sum()), perf_did=a[0], perf_lo=a[1], perf_hi=a[2],
                       lpts_did=b[0], lpts_lo=b[1], lpts_hi=b[2]))
TT = pd.DataFrame(types_)
display(TT.round(3))
TT.to_csv(os.path.join(OUT, "new_coach_type.csv"), index=False)
RES["new_coach_type"] = TT.to_dict(orient="records")
tsn = pd.DataFrame([dict(league=l, season=se, team=t, theta=100 * v) for (l, se), d in THETA.items() for t, v in d.items()])
tsn["ys"] = tsn["season"].str[:4].astype(int)
nx = tsn.assign(ys=tsn["ys"] - 1).rename(columns={"theta": "theta_next"})[["league", "ys", "team", "theta_next"]]
pv = tsn.assign(ys=tsn["ys"] + 1).rename(columns={"theta": "theta_prev"})[["league", "ys", "team", "theta_prev"]]
PN = tsn.merge(nx, on=["league", "ys", "team"], how="left").merge(pv, on=["league", "ys", "team"], how="left")
changed_keys = set(zip(events["league"], events["season"], events["team"]))
PN["changed"] = [(l, se, t) in changed_keys for l, se, t in zip(PN["league"], PN["season"], PN["team"])]
PN = PN[PN["theta_next"].notna()].copy()
PN["prev0"], PN["prev_na"] = PN["theta_prev"].fillna(0), PN["theta_prev"].isna().astype(int)
m_next = smf.ols("theta_next ~ changed + theta + prev0 + prev_na + C(league)", data=PN).fit(cov_type="HC1")
ci_n = m_next.conf_int().loc["changed[T.True]"]
RES["next_season"] = dict(effect=r3(m_next.params["changed[T.True]"]), lo=r3(ci_n[0]), hi=r3(ci_n[1]),
                          n=len(PN), n_changed=int(PN["changed"].sum()))
print("Next-season strength:", RES["next_season"])


## 10. Minimum detectable effects (sample size justification)
The smallest true effect that a two-sided test at the 5% level detects with 80% power is 2.80 standard errors.

In [ ]:
Z_MDE = norm.ppf(0.975) + norm.ppf(0.80)
RES["mde"] = dict(z=r3(Z_MDE), n_changes=len(R_main),
                  did_perf_pp=r3(Z_MDE * did(R_main, "perf")[3]), did_lpts=r3(Z_MDE * did(R_main, "lpts")[3]),
                  n_team_seasons=len(HALF), corr=r3(np.tanh(Z_MDE / np.sqrt(len(HALF) - 3))))
print(RES["mde"])


## 11. Save

In [ ]:
with open(os.path.join(OUT, "results.json"), "w") as f:
    json.dump(RES, f, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
zpath = os.path.join(WORK, "coach_analysis_out.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(OUT):
        z.write(os.path.join(OUT, fn), fn)
print("Saved", zpath)
try:
    from google.colab import files
    files.download(zpath)
except Exception:
    pass
